# Embedding Cliffs

Probing what single embedding dimensions and word-difference directions do inside a running GGUF model, by injecting perturbed vectors via the raw `llama_batch` API and asking the model itself to explain what changed.

One cell per use case. Run the **Setup** cell first; everything after reuses `embd_probe.py`'s functions directly instead of re-implementing the ctypes plumbing.

Swap `MODEL_PATH` for a bigger instruct model (e.g. Llama-3.1-8B-Instruct) to reproduce the sharper results noted in a few cells below.

In [ ]:
import sys
sys.path.insert(0, ".")
import embd_probe as m
from llama_cpp import Llama

MODEL_PATH = "/home/victor/text-generation-webui/user_data/models/Llama-3.2-1B-Instruct-Q8_0.gguf"

# logits_all=True works around a NULL-logits bug when driving llama_decode manually -- see embd_probe.py
llm = Llama(model_path=MODEL_PATH, n_ctx=2048, n_gpu_layers=0, logits_all=True, verbose=False)
ctx = llm.ctx
n_vocab = llm.n_vocab()
n_embd = llm.n_embd()
eos_id = llm.token_eos()
template = m.get_chat_template(llm)          # None if the GGUF has no chat_template metadata
embed_table = m.load_token_embeddings(MODEL_PATH)   # (n_vocab, n_embd) float32, dequantized straight from the GGUF

print(f"n_vocab={n_vocab} n_embd={n_embd} chat_template={'yes' if template else 'no'}")


def ask(prompt, target=None, perturb_fn=None, n_predict=50):
    """Render `prompt` through the chat template, optionally perturbing just the
    substring `target` with perturb_fn(vecs) -> vecs, and generate a continuation.
    perturb_fn=None runs the plain, unperturbed baseline."""
    token_ids = m.tokenize_prompt(llm, template, prompt)
    content_range = None
    if target is not None:
        content_ids = llm.tokenize(prompt.encode("utf-8"), add_bos=False, special=False)
        span = m.find_subsequence(token_ids, content_ids)
        target_ids = llm.tokenize(target.encode("utf-8"), add_bos=False, special=False)
        rel = m.find_subsequence(content_ids, target_ids)
        content_range = (span[0] + rel[0], span[0] + rel[1])
    m._kv_cache_clear(ctx)
    if perturb_fn is None:
        n_past = m.decode_tokens(ctx, token_ids, 0)
    else:
        n_past = m.decode_perturbed(ctx, embed_table, token_ids, n_embd, perturb_fn, content_range)
    return m.generate(llm, ctx, n_vocab, n_past, n_predict, eos_id)


def explain(full_text_a, full_text_b, n_predict=70):
    """Ask the model, unperturbed, to explain the difference between two full texts."""
    return m.explain_diff(llm, ctx, n_vocab, template, full_text_a, full_text_b, n_predict, eos_id)


## Use case: cosine similarity sanity check

Before trusting any direction math, confirm the geometry behaves sensibly. Note the leading space on each word -- BPE tokenizes `"tiger"` and `" tiger"` as different tokens; only the space-prefixed, word-initial form is the clean single-token comparison.

In [ ]:
m.print_cosine(llm, embed_table, " cat", " dog")
print()
m.print_cosine(llm, embed_table, " cat", " car")
print()
m.print_cosine(llm, embed_table, " cat", " tiger")


## Use case: are raw dimensions interpretable?

For a handful of dimensions, list the vocabulary's highest- and lowest-scoring tokens on that axis. Spoiler: no clean single-concept clusters turn up -- consistent with superposition (dense embeddings pack many entangled features per axis).

In [ ]:
for idx in [0, 100, 500, 1000, 2000]:
    m.print_top_tokens(llm, embed_table, idx, 10)


## Use case: dialing "hackiness" on the word "Fix"

Direction: mean(*hack, kludge, quick, patch, hacky*) minus mean(*clean, proper, robust, elegant, correct*), injected into a single occurrence of the word "Fix" in a plain question. Sweep the scale and watch two distinct effects either side of zero: negative scale flips *which word the model thinks it read*; positive scale keeps "fix" but shifts its connotation toward makeshift/informal.

In [ ]:
hack_clean_direction = m.diff_mean_direction(
    llm, embed_table,
    [" hack", " kludge", " quick", " patch", " hacky"],
    [" clean", " proper", " robust", " elegant", " correct"],
)

prompt = "What does the word Fix mean to you? Explain in one sentence."
baseline = ask(prompt)
print(f"baseline: {baseline}\n")

for scale in [-9, -6, -3, 3, 8, 10]:
    out = ask(prompt, target=" Fix", perturb_fn=m.make_direction_fn(hack_clean_direction, scale))
    print(f"scale {scale:+.1f}: {out}")


## Use case: `--compare` -- have the model explain the nuance

Same axis, scale 3. Instead of eyeballing the output, ask the model (unperturbed) to explain the difference between the baseline and perturbed continuations.

In [ ]:
scale = 3
perturbed = ask(prompt, target=" Fix", perturb_fn=m.make_direction_fn(hack_clean_direction, scale))
print(f"perturbed (scale {scale}): {perturbed}\n")

explanation = explain(prompt + baseline, prompt + perturbed)
print("model's explanation of the difference:")
print(explanation)


## Use case: mapping the coherence cliff

Direction: mean(*lion, bear, tiger*) minus mean(*bunny, gazelle, deer*), on the word "Fish". The transition from one stable identity to another isn't a smooth slope -- there's a narrow, genuinely garbled band between two coherent basins. Finer-grained scales here narrow in on exactly where it sits.

In [ ]:
predator_prey_direction = m.diff_mean_direction(
    llm, embed_table,
    [" lion", " bear", " tiger"],
    [" bunny", " gazelle", " deer"],
)

prompt = "What does the word Fish mean to you? Explain in one sentence."
for scale in [-3, -1.5, 0, 1.5, 2.0, 2.25, 2.5, 2.75, 3]:
    out = ask(prompt, target=" Fish", perturb_fn=m.make_direction_fn(predator_prey_direction, scale))
    print(f"scale {scale:+.2f}: {out}")


## Use case: unrelated word exposes the real mechanism

Apply the *same* predator/prey direction to a word that has nothing to do with animals: "Boat". If the technique were adding a soft "predator-ish" connotation, "boat" should stay recognizable with a shifted flavor. Instead, at a strong enough scale, the perturbation completely overrides token identity -- no trace of "boat" survives. This is the finding that on-topic target words (like "Fish" above) can look like graceful connotation shifts purely because the override happens to land on a plausible neighbor.

In [ ]:
prompt = "What does the word Boat mean to you? Explain in one sentence."
for scale in [-3, 0, 3]:
    out = ask(prompt, target=" Boat", perturb_fn=m.make_direction_fn(predator_prey_direction, scale))
    print(f"scale {scale:+.1f}: {out}")


## Use case: an axis that only resolves on one side

Direction: mean(*tower, whale*) minus mean(*mouse, flea, atom*) ("big" vs "small"), on "Fish" again. The small-group side eventually locks in cleanly by scale -4/-5. The big-group side never resolves even out to scale 5 -- it stays stuck in a garbled near-miss basin, plausibly because "tower" and "whale" are semantically distant from each other, so their mean sits between concepts rather than at one.

In [ ]:
big_small_direction = m.diff_mean_direction(
    llm, embed_table,
    [" tower", " whale"],
    [" mouse", " flea", " atom"],
)

prompt = "What does the word Fish mean to you? Explain in one sentence."
for scale in [-5, -4, -3, 0, 3, 4, 5]:
    out = ask(prompt, target=" Fish", perturb_fn=m.make_direction_fn(big_small_direction, scale))
    print(f"scale {scale:+.1f}: {out}")
